In [ ]:
# ================================================================
# PRACTICAL 9: STRIPS AND GOAL-STACK PLANNING
# Artificial Intelligence Laboratory
# Problem: Robot Navigation from Room A to Room C
# Environment: Google Colab
# Dependencies: Python standard library only
# ================================================================

from copy import deepcopy
from pprint import pprint


# ================================================================
# STEP 1: DEFINE THE PLANNING PROBLEM
# ================================================================

print("=" * 70)
print("PRACTICAL 9: STRIPS / GOAL-STACK PLANNING")
print("=" * 70)

# Initial state: Robot is in Room A
INITIAL_STATE = {"At_A"}

# Goal state: Robot must reach Room C
GOAL = "At_C"

# The robot can move along the following route:
# Room A ---> Room B ---> Room C

print("\nSTEP 1: PROBLEM DEFINITION")
print("-" * 70)
print("Initial state:", INITIAL_STATE)
print("Goal:", GOAL)
print("Environment: Room A -> Room B -> Room C")


# ================================================================
# STEP 2: DEFINE STRIPS ACTIONS
# ================================================================
# Each action contains:
# 1. Preconditions: facts that must already be true
# 2. Add effects: facts that become true
# 3. Delete effects: facts that become false
#
# State transition:
# NEW STATE = (OLD STATE - DELETE EFFECTS) UNION ADD EFFECTS
# ================================================================

ACTIONS = [
    {
        "name": "Move A to B",
        "preconditions": {"At_A"},
        "add_effects": {"At_B"},
        "delete_effects": {"At_A"}
    },
    {
        "name": "Move B to C",
        "preconditions": {"At_B"},
        "add_effects": {"At_C"},
        "delete_effects": {"At_B"}
    }
]


def display_actions(actions):
    """Display all action schemas in STRIPS format."""

    print("\nSTEP 2: STRIPS ACTION REPRESENTATION")
    print("-" * 70)

    for action in actions:
        print(f"\nAction: {action['name']}")
        print(" Preconditions :", action["preconditions"])
        print(" Add effects :", action["add_effects"])
        print(" Delete effects:", action["delete_effects"])


display_actions(ACTIONS)


# ================================================================
# STEP 3: FIND ACTIONS THAT CAN ACHIEVE A GOAL
# ================================================================

def find_actions_for_goal(goal, actions):
    """
    Return actions whose add effects contain the requested goal.
    """

    return [
        action
        for action in actions
        if goal in action["add_effects"]
    ]


# ================================================================
# STEP 4: APPLY AN ACTION SAFELY
# ================================================================

def apply_action(state, action):
    """
    Apply a STRIPS action if all its preconditions are satisfied.

    Returns a NEW state without modifying the input state.
    Raises ValueError if the action is not applicable.
    """

    missing = action["preconditions"] - state

    if missing:
        raise ValueError(
            f"Action '{action['name']}' cannot execute. "
            f"Missing preconditions: {missing}"
        )

    new_state = (
        state - action["delete_effects"]
    ) | action["add_effects"]

    return new_state


# ================================================================
# STEP 5: GOAL-STACK-STYLE BACKWARD REASONING
# ================================================================
# The planner:
# 1. Checks whether the goal is already true.
# 2. Selects an action that can achieve the goal.
# 3. Recursively achieves that action's preconditions.
# 4. Checks the preconditions before execution.
# 5. Applies the action and records it in the plan.
#
# IMPORTANT:
# This is an educational recursive subgoal planner.
# It demonstrates backward reasoning but is not a complete
# classical STRIPS planner with general backtracking.
# ================================================================

def achieve_goal(
    goal,
    state,
    actions,
    plan,
    active_goals=None,
    depth=0,
    verbose=True
):
    """
    Attempt to achieve a goal using recursive subgoal reasoning.

    Parameters
    ----------
    goal : str
        The proposition to make true.
    state : set
        The current mutable planning state.
    actions : list
        Available STRIPS actions.
    plan : list
        Actions appended in execution order.
    active_goals : set
        Goals currently being explored; used for cycle detection.
    depth : int
        Recursion depth for readable tracing.
    verbose : bool
        Whether to display the reasoning trace.

    Returns
    -------
    bool
        True if this goal was achieved; otherwise False.
    """

    if active_goals is None:
        active_goals = set()

    indent = " " * depth

    if verbose:
        print(f"\n{indent}Goal: {goal}")
        print(f"{indent}Current state: {sorted(state)}")

    # CASE 1: Goal already achieved
    if goal in state:
        if verbose:
            print(f"{indent}Goal already satisfied.")
        return True

    # CASE 2: Detect recursive goal cycles
    if goal in active_goals:
        if verbose:
            print(
                f"{indent}Cycle detected while pursuing "
                f"{goal}. Returning failure."
            )
        return False

    # CASE 3: Find an action that establishes the goal
    candidates = find_actions_for_goal(goal, actions)

    if not candidates:
        if verbose:
            print(f"{indent}No action can achieve {goal}.")
        return False

    # This demonstration selects the first candidate.
    # General planning requires searching alternatives if necessary.
    action = candidates[0]

    if verbose:
        print(f"{indent}Selected action: {action['name']}")

    active_goals.add(goal)

    try:
        # First achieve all action preconditions
        for precondition in sorted(action["preconditions"]):

            if verbose:
                print(
                    f"{indent}Need to establish precondition: "
                    f"{precondition}"
                )

            success = achieve_goal(
                precondition,
                state,
                actions,
                plan,
                active_goals,
                depth + 1,
                verbose
            )

            if not success:
                if verbose:
                    print(
                        f"{indent}Could not achieve "
                        f"precondition {precondition}."
                    )
                return False

        # Verify preconditions immediately before action execution
        missing = action["preconditions"] - state

        if missing:
            if verbose:
                print(
                    f"{indent}Action failed: unsatisfied "
                    f"preconditions {missing}"
                )
            return False

        # Apply the action
        old_state = set(state)
        new_state = apply_action(state, action)

        state.clear()
        state.update(new_state)

        # Record action in forward execution order
        plan.append(action["name"])

        if verbose:
            print(f"{indent}EXECUTING: {action['name']}")
            print(f"{indent}State before: {sorted(old_state)}")
            print(f"{indent}State after : {sorted(state)}")

        return goal in state

    finally:
        active_goals.remove(goal)


# ================================================================
# STEP 6: RUN THE PLANNER
# ================================================================

print("\n\nSTEP 3: BACKWARD GOAL REASONING")
print("=" * 70)

current_state = deepcopy(INITIAL_STATE)
plan = []

planning_success = achieve_goal(
    goal=GOAL,
    state=current_state,
    actions=ACTIONS,
    plan=plan,
    verbose=True
)


# ================================================================
# STEP 7: DISPLAY THE GENERATED PLAN
# ================================================================

print("\n\nSTEP 4: GENERATED ACTION PLAN")
print("=" * 70)

if planning_success:

    for step_number, action_name in enumerate(plan, start=1):
        print(f"Step {step_number}: {action_name}")

    print("\nPlanner's final state:", sorted(current_state))

else:
    print("Planning failed.")
    print("The goal could not be achieved with the available actions.")


# ================================================================
# STEP 8: INDEPENDENT PLAN VALIDATION
# ================================================================
# Replay every action from the ORIGINAL initial state.
# Verify each action's preconditions and the final goal.
# ================================================================

def validate_plan(initial_state, plan, actions, goal):
    """Replay and independently validate a generated action plan."""

    print("\n\nSTEP 5: INDEPENDENT PLAN VALIDATION")
    print("=" * 70)

    replay_state = set(initial_state)

    action_lookup = {
        action["name"]: action
        for action in actions
    }

    print("Starting state:", sorted(replay_state))

    for step_number, action_name in enumerate(plan, start=1):

        if action_name not in action_lookup:
            print("ERROR: Unknown action:", action_name)
            return False, replay_state

        action = action_lookup[action_name]

        # Every action must be applicable
        missing = action["preconditions"] - replay_state

        if missing:
            print(f"ERROR at step {step_number}: {action_name}")
            print("Missing preconditions:", missing)
            return False, replay_state

        # Apply the action independently
        replay_state = apply_action(replay_state, action)

        print(
            f"Step {step_number}: {action_name}"
            f" -> State: {sorted(replay_state)}"
        )

    goal_achieved = goal in replay_state

    print("\nFinal replay state:", sorted(replay_state))
    print("Goal achieved:", goal_achieved)

    return goal_achieved, replay_state


if planning_success:

    validation_success, replay_final_state = validate_plan(
        initial_state=INITIAL_STATE,
        plan=plan,
        actions=ACTIONS,
        goal=GOAL
    )

else:
    validation_success = False
    replay_final_state = set(INITIAL_STATE)


# ================================================================
# STEP 9: FINAL SUMMARY
# ================================================================

print("\n\nFINAL PRACTICAL SUMMARY")
print("=" * 70)

print("Initial state :", sorted(INITIAL_STATE))
print("Required goal :", GOAL)
print("Planning successful :", planning_success)
print("Validation successful :", validation_success)
print("Final state :", sorted(replay_final_state))

if planning_success and validation_success:
    print("\nRESULT: SUCCESS")
    print("The robot has reached its target room.")
    print("All action preconditions were satisfied.")
    print("The generated plan passed independent validation.")
else:
    print("\nRESULT: FAILURE")
    print("The goal was not reached or the plan was invalid.")

# Assertions make the practical testable
assert planning_success, "The planner failed to produce a plan."
assert validation_success, "Independent validation failed."
assert GOAL in replay_final_state, "The target goal was not achieved."
assert replay_final_state == current_state, (
    "The planner's final state differs from the replayed final state."
)

print("\nAll automated checks passed.")
print("=" * 70)


# ================================================================
# STEP 10: STUDENT EXPERIMENTS
# ================================================================
# Try these after the first successful run:
#
# Experiment A:
# Change INITIAL_STATE to {"At_B"}.
# Expected plan: Move B to C.
#
# Experiment B:
# Remove "Move A to B" from ACTIONS.
# Starting at At_A, the goal At_C should be unreachable
# with the remaining actions.
#
# Experiment C:
# Add an action that moves directly from A to C:
#
# {
# "name": "Move A to C",
# "preconditions": {"At_A"},
# "add_effects": {"At_C"},
# "delete_effects": {"At_A"}
# }
#
# Compare the resulting plan.
#
# IMPORTANT:
# After changing the problem, rerun the entire cell so all states,
# plans, and validation results are recomputed.
# ================================================================

PRACTICAL 9: STRIPS / GOAL-STACK PLANNING

STEP 1: PROBLEM DEFINITION
----------------------------------------------------------------------
Initial state: {'At_A'}
Goal: At_C
Environment: Room A -> Room B -> Room C

STEP 2: STRIPS ACTION REPRESENTATION
----------------------------------------------------------------------

Action: Move A to B
 Preconditions : {'At_A'}
 Add effects : {'At_B'}
 Delete effects: {'At_A'}

Action: Move B to C
 Preconditions : {'At_B'}
 Add effects : {'At_C'}
 Delete effects: {'At_B'}


STEP 3: BACKWARD GOAL REASONING

Goal: At_C
Current state: ['At_A']
Selected action: Move B to C
Need to establish precondition: At_B

 Goal: At_B
 Current state: ['At_A']
 Selected action: Move A to B
 Need to establish precondition: At_A

  Goal: At_A
  Current state: ['At_A']
  Goal already satisfied.
 EXECUTING: Move A to B
 State before: ['At_A']
 State after : ['At_B']
EXECUTING: Move B to C
State before: ['At_B']
State after : ['At_C']


STEP 4: GENERATED ACTION PLAN
